In [2]:
import boto3, random, base64, json, io
from PIL import Image
ENDPOINT_NAME = 'YOLOv8-CFU-SageMaker-endpoint' 

In [3]:
orig_image = Image.open('cfu_positive.jpg')

In [4]:
# Calculate the parameters for image resizing
image_height, image_width = orig_image.size
model_height, model_width = 640, 640
x_ratio = image_width/model_width
y_ratio = image_height/model_height
orig_image.thumbnail((model_height, model_width), Image.Resampling.LANCZOS)

In [5]:
# Convert the image to the jpeg and save the jpeg as a byte stream 
img_byte_arr = io.BytesIO()
orig_image.save(img_byte_arr, format='jpeg')

In [6]:
# Convert tyhe bytes intoe base64
payload = base64.b64encode(img_byte_arr.getvalue())

In [7]:
runtime= boto3.client('runtime.sagemaker')
response = runtime.invoke_endpoint(EndpointName=ENDPOINT_NAME,
                                        ContentType='text/csv',
                                        Body=payload)
response_body = response['Body'].read()
result = json.loads(response_body.decode('ascii'))

In [8]:
result

{'boxes': [[506.3719482421875,
   180.7960205078125,
   571.8616943359375,
   249.982421875,
   0.9310478568077087,
   2.0],
  [156.88983154296875,
   152.29501342773438,
   217.8939208984375,
   215.78219604492188,
   0.928846538066864,
   2.0],
  [323.830810546875,
   93.73703002929688,
   382.439697265625,
   152.0107421875,
   0.8800340890884399,
   2.0],
  [74.50230407714844,
   245.63987731933594,
   131.2914581298828,
   365.6236572265625,
   0.8417573571205139,
   4.0],
  [392.492431640625,
   109.95610046386719,
   472.2001953125,
   172.9574737548828,
   0.8240453600883484,
   5.0],
  [359.82037353515625,
   473.3492431640625,
   468.3182373046875,
   553.0460205078125,
   0.81979900598526,
   8.0]]}

In [27]:

if 'boxes' in result:
    for idx,(x1,y1,x2,y2,conf,lbl) in enumerate(result['boxes']):
        # Draw Bounding Boxes
        x1, x2 = int(x_ratio*x1), int(x_ratio*x2)
        y1, y2 = int(y_ratio*y1), int(y_ratio*y2)
        color = (random.randint(10,255), random.randint(10,255), random.randint(10,255))
        cv2.rectangle(orig_image, (x1,y1), (x2,y2), color, 4)
        cv2.putText(orig_image, f"Class: {int(lbl)}", (x1,y1-40), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2, cv2.LINE_AA)
        cv2.putText(orig_image, f"Conf: {int(conf*100)}", (x1,y1-10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2, cv2.LINE_AA)

error: OpenCV(4.10.0) :-1: error: (-5:Bad argument) in function 'rectangle'
> Overload resolution failed:
>  - img is not a numpy array, neither a scalar
>  - Expected Ptr<cv::UMat> for argument 'img'
>  - img is not a numpy array, neither a scalar
>  - Expected Ptr<cv::UMat> for argument 'img'


In [13]:
im_w_boxes = cv2.cvtColor(orig_image, cv2.COLOR_BGR2RGB)
result['image'] = base64.b64encode(im_w_boxes).decode('utf-8')

In [ ]:
json.dumps(result)